In [ ]:
# Notebook to explore and viz inference abilities of WiLoR (https://github.com/rolpotamias/WiLoR)

In [ ]:
from pathlib import Path
import torch
import argparse
import os
import cv2
import numpy as np
import pandas as pd
import json
from typing import Dict, Optional

from wilor.models import WiLoR, load_wilor
from wilor.utils import recursive_to
from wilor.datasets.vitdet_dataset import ViTDetDataset, DEFAULT_MEAN, DEFAULT_STD
from wilor.utils.renderer import Renderer, cam_crop_to_full
from ultralytics import YOLO 
LIGHT_PURPLE=(0.25098039,  0.274117647,  0.65882353)

%matplotlib ipympl 

# These are command-line arguments in the script version
img_folder = "images" # Folder with input images
out_folder = "out_demo" # Output folder to save rendered results
save_mesh = False # If set, save meshes to disk also
rescale_factor = 2.0 # Factor for padding the bbox
file_type = ['*.jpg', '*.png', '*.jpeg'] # List of file extensions to consider

def project_full_img(points, cam_trans, focal_length, img_res): 
    camera_center = [img_res[0] / 2., img_res[1] / 2.]
    K = torch.eye(3) 
    K[0,0] = focal_length
    K[1,1] = focal_length
    K[0,2] = camera_center[0]
    K[1,2] = camera_center[1]
    points = points + cam_trans
    points = points / points[..., -1:] 
    
    V_2d = (K @ points.T).T 
    return V_2d[..., :-1]

In [ ]:
model, model_cfg = load_wilor(checkpoint_path = './pretrained_models/wilor_final.ckpt' , cfg_path= './pretrained_models/model_config.yaml')
detector = YOLO('./pretrained_models/detector.pt')
# Setup the renderer
renderer = Renderer(model_cfg, faces=model.mano.faces)
#renderer_side = Renderer(model_cfg, faces=model.mano.faces)

device   = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model    = model.to(device)
detector = detector.to(device)
model.eval()

# Make output directory if it does not exist
os.makedirs(out_folder, exist_ok=True)

In [ ]:
from matplotlib import pyplot as plt

HAND_21_KEYPOINTS = [
    "ulnar_palm",  # 1
    "radial_palm",  # 2
    "thumb_metacarpal",  # 3
    "thumb_proximal",  # 4
    "thumb_distal",  # 5
    "index_metacarpal",  # 6
    "index_proximal",  # 7
    "index_middle",  # 8
    "index_distal",  # 9
    "middle_metacarpal",  # 10
    "middle_proximal",  # 11
    "middle_middle",  # 12
    "middle_distal",  # 13
    "ring_metacarpal",  # 14
    "ring_proximal",  # 15
    "ring_middle",  # 16
    "ring_distal",  # 17
    "pinkie_metacarpal",  # 18
    "pinkie_proximal",  # 19
    "pinkie_middle",  # 20
    "pinkie_distal",  # 21
]

HAND_21_SKELETON = [
    (1, 2),
    (1, 18),
    (2, 3),
    (3, 4),
    (3, 6),
    (4, 5),
    (6, 10),
    (6, 7),
    (7, 8),
    (8, 9),
    (10, 14),
    (10, 11),
    (11, 12),
    (12, 13),
    (14, 18),
    (14, 15),
    (15, 16),
    (16, 17),
    (18, 19),
    (19, 20),
    (20, 21),
]

def plot_hand_joints(joints):
    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    
    joints_3d = np.array(joints).flatten().reshape(21,3)
    
    #plt.scatter(joints_3d[:,0], joints_3d[:,1], joints_3d[:,2])
    
    for i, seg in enumerate(HAND_21_SKELETON):
        plt.plot([joints_3d[seg[0] - 1][0], joints_3d[seg[1] - 1][0]], [joints_3d[seg[0] - 1][1], joints_3d[seg[1] - 1][1]], [joints_3d[seg[0] - 1][2], joints_3d[seg[1] - 1][2]], 'b-')
    
    plt.show()

def draw_hand_joints_on_image(joints, image, bbox=None):

    if bbox is None:
        xmin = np.min(joints[:,0])
        xmax = np.max(joints[:,0])
        ymin = np.min(joints[:,1])
        ymax = np.max(joints[:,1])
    else:
        xmin, ymin, xmax, ymax = bbox

    width = xmax - xmin
    height = ymax - ymin
    
    fig = plt.figure()
    ax = fig.add_subplot()
    
    ax.imshow(image)
    
    ax.add_patch(plt.Rectangle((xmin, ymin), width, height, color="green", fill=False))
    
    ax.scatter(joints[:,0], joints[:,1], s=1)
    
    for i, seg in enumerate(HAND_21_SKELETON):
        plt.plot([joints[seg[0] - 1][0], joints[seg[1] - 1][0]], [joints[seg[0] - 1][1], joints[seg[1] - 1][1]], 'b-')
    
    plt.show()

In [ ]:
#img_path = "demo_img/Mona_Lisa_all.jpg"
img_path = "demo_img/test8.jpg"

img_cv2 = cv2.imread(str(img_path))

img2 = img_cv2[:,:,::-1]

detections = detector(img_cv2, conf = 0.3, verbose=False)[0]
bboxes    = []
is_right  = []

for det in detections: 
    Bbox = det.boxes.data.cpu().detach().squeeze().numpy()
    is_right.append(det.boxes.cls.cpu().detach().squeeze().item())
    bboxes.append(Bbox[:4].tolist())

if len(bboxes) != 0:
    boxes = np.stack(bboxes)
    right = np.stack(is_right)
    dataset = ViTDetDataset(model_cfg, img_cv2, boxes, right, rescale_factor=rescale_factor)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)

    all_verts = []
    all_cam_t = []
    all_right = []
    all_joints= []
    all_kpts  = []
    all_joints_2d = []

    for batch in dataloader: 
        batch = recursive_to(batch, device)

        with torch.no_grad():
            out = model(batch) 
            
        multiplier    = (2*batch['right']-1)
        pred_cam      = out['pred_cam']
        pred_cam[:,1] = multiplier*pred_cam[:,1]
        box_center    = batch["box_center"].float()
        box_size      = batch["box_size"].float()
        img_size      = batch["img_size"].float()
        scaled_focal_length = model_cfg.EXTRA.FOCAL_LENGTH / model_cfg.MODEL.IMAGE_SIZE * img_size.max()
        pred_cam_t_full     = cam_crop_to_full(pred_cam, box_center, box_size, img_size, scaled_focal_length).detach().cpu().numpy()
        
        # Render the result
        batch_size = batch['img'].shape[0]
        for n in range(batch_size):
            # Get filename from path img_path
            img_fn, _ = os.path.splitext(os.path.basename(img_path))
            
            verts  = out['pred_vertices'][n].detach().cpu().numpy()
            joints = out['pred_keypoints_3d'][n].detach().cpu().numpy()
            
            is_right    = batch['right'][n].cpu().numpy()
            verts[:,0]  = (2*is_right-1)*verts[:,0]
            joints[:,0] = (2*is_right-1)*joints[:,0]
            cam_t = pred_cam_t_full[n]
            kpts_2d = project_full_img(verts, cam_t, scaled_focal_length, img_size[n])

            joints_2d = project_full_img(joints, cam_t, scaled_focal_length, img_size[n])
            
            all_verts.append(verts)
            all_cam_t.append(cam_t)
            all_right.append(is_right)
            all_joints.append(joints)
            all_kpts.append(kpts_2d)
            all_joints_2d.append(joints_2d)

            print(img_fn, "hand", n, "right?", is_right)
            # print("# VERTS:", len(verts))
            # print("CAM:", cam_t)
            # print("RIGHT:", is_right)
            # print("# JOINTS:", len(joints))
            # print("# KPTS:", len(kpts_2d))

            global_orient = out["pred_mano_params"]["global_orient"].cpu().tolist()[n][0]
            
            global_joints = np.matmul(joints, global_orient).flatten()
            
            plot_hand_joints(joints)
            draw_hand_joints_on_image(joints_2d.cpu().numpy(), img2, bboxes[n])
            plot_hand_joints(global_joints)
            
            # Save all meshes to disk
            if save_mesh:
                camera_translation = cam_t.copy()
                tmesh = renderer.vertices_to_trimesh(verts, camera_translation, LIGHT_PURPLE, is_right=is_right)
                tmesh.export(os.path.join(out_folder, f'{img_fn}_{n}.obj'))

    # # Render front view
    # if len(all_verts) > 0:
    #     misc_args = dict(
    #         mesh_base_color=LIGHT_PURPLE,
    #         scene_bg_color=(1, 1, 1),
    #         focal_length=scaled_focal_length,
    #     )
    #     cam_view = renderer.render_rgba_multiple(all_verts, cam_t=all_cam_t, render_res=img_size[n], is_right=all_right, **misc_args)

    #     # Overlay image
    #     input_img = img_cv2.astype(np.float32)[:,:,::-1]/255.0
    #     input_img = np.concatenate([input_img, np.ones_like(input_img[:,:,:1])], axis=2) # Add alpha channel
    #     input_img_overlay = input_img[:,:,:3] * (1-cam_view[:,:,3:]) + cam_view[:,:,:3] * cam_view[:,:,3:]

    #     cv2.imwrite(os.path.join(out_folder, f'{img_fn}.jpg'), 255*input_img_overlay[:, :, ::-1])

In [ ]:
# xywh seems to be incorrectly positioned for some reason (!)
xywh = detections[0].boxes.xywh.cpu().numpy()[0]
# but xyxy is correct (!)
xyxy = detections[0].boxes.xyxy.cpu().numpy()[0]

fig = plt.figure()
ax = fig.add_subplot()

ax.imshow(img2)

#ax.add_patch(plt.Rectangle((xywh[0], xywh[1]), xywh[2], xywh[3], color="green", fill=False))

ax.add_patch(plt.Rectangle((xyxy[0], xyxy[1]), abs(xyxy[0] - xyxy[2]), abs(xyxy[1] - xyxy[3]), color="green", fill=False))

ax.scatter(all_joints_2d[0][:,0], all_joints_2d[0][:,1], s=1)

plt.show()